# PRUEBAS

### Parámetros Comunes

In [1]:
import pandas as pd
import time
from datetime import datetime

from typing import Iterable, Optional, Tuple, Dict, Any
import datetime as dt

# Cargar configuración DINAMIDA de acuerdo al entorno
from dotenv import dotenv_values
import os
import sys

# Acceso a Datos
import psycopg2 as pg2
import pyodbc
import sqlalchemy
from sqlalchemy import text

# Determinar la ruta base del proyecto
# BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
BASE_DIR = "C:/ETL/FORECAST"  # Ruta fija para desarrollo local
CORE_DIR = os.path.join(BASE_DIR, 'forecast_core')
sys.path.insert(0, CORE_DIR)
print("Contenido de sys.path:")
for path in sys.path:
    print(path)

ENV_PATH = os.environ.get("FORECAST_ENV_PATH", "C:/ETL/FORECAST/.env")  # Toma Producción si está definido, o la ruta por defecto
if not os.path.exists(ENV_PATH):
    print(f"El archivo .env no existe en la ruta: {ENV_PATH}")
    print(f"Directorio actual: {os.getcwd()}")
    sys.exit(1)
    
secrets = dotenv_values(ENV_PATH)
folder = f"{secrets['BASE_DIR']}/{secrets['FOLDER_DATOS']}"

# Verificar en que entorno está funcioando
print(f"Python executable: {sys.executable}")
print(f"PATH: {os.environ.get('PATH')}")


Contenido de sys.path:
C:/ETL/FORECAST\forecast_core
C:\Program Files\Python312\python312.zip
C:\Program Files\Python312\DLLs
C:\Program Files\Python312\Lib
C:\Program Files\Python312
c:\ETL\venv

c:\ETL\venv\Lib\site-packages
c:\ETL\venv\Lib\site-packages\win32
c:\ETL\venv\Lib\site-packages\win32\lib
c:\ETL\venv\Lib\site-packages\Pythonwin
Python executable: c:\ETL\venv\Scripts\python.exe
PATH: c:\ETL\venv\Scripts;C:\ETL\venv\Scripts;C:\Program Files\Python312\Scripts\;C:\Program Files\Python312\;C:\windows\system32;C:\windows;C:\windows\System32\Wbem;C:\windows\System32\WindowsPowerShell\v1.0\;C:\windows\System32\OpenSSH\;C:\Program Files\Git\cmd;C:\Program Files\HP\HP One Agent;C:\Program Files\Microsoft SQL Server\150\Tools\Binn\;C:\Program Files\Microsoft SQL Server\Client SDK\ODBC\170\Tools\Binn\;C:\Program Files (x86)\Microsoft SQL Server\160\DTS\Binn\;C:\WINDOWS\system32;C:\WINDOWS;C:\WINDOWS\System32\Wbem;C:\WINDOWS\System32\WindowsPowerShell\v1.0\;C:\WINDOWS\System32\OpenSSH\

## Funciones Comunes

In [ ]:
# Solo importa lo necesario desde el módulo de funciones
# from forecast_core.funciones_forecast import (
#     # get_forecast,
#     generar_datos,
#     Procesar_ALGO_01,
#     Procesar_ALGO_02,
#     Procesar_ALGO_03,
#     Procesar_ALGO_04,
#     Procesar_ALGO_05,
#     Procesar_ALGO_06,
#     Procesar_ALGO_07,     
#     generar_datos,    
#     get_execution_execute_by_status,
#     get_full_parameters,
#     update_execution_execute
# )


def Close_Connection(conn): 
    if conn is not None:
        conn.close()
        # print("✅ Conexión cerrada.")    
    return True

def Open_Conn_Postgres():
    conn_str = f"dbname={secrets['PGP_DB']} user={secrets['PGP_USER']} password={secrets['PGP_PASSWORD']} host={secrets['PGP_HOST']} port={secrets['PGP_PORT']}"
    for i in range(5):
        try:
            conn = pg2.connect(conn_str)
            return conn 
        except Exception as e:
            print(f"Error en la conexión, intento {i+1}/{5}: {e}")
            time.sleep(5)
    return None  # Retorna None si todos los intentos fallan

def Open_Diarco_Postgres():
    conn_str = f"dbname={secrets['PG_DB']} user={secrets['PG_USER']} password={secrets['PG_PASSWORD']} host={secrets['PG_HOST']} port={secrets['PG_PORT']}"
    for i in range(5):
        try:
            conn = pg2.connect(conn_str)
            return conn 
        except Exception as e:
            print(f"Error en la conexión, intento {i+1}/{5}: {e}")
            time.sleep(5)
    return None  # Retorna None si todos los intentos fallan

In [ ]:
def extender_datos_forecast(algoritmo, name, id_proveedor):
    # Recuperar Historial de Ventas
    df_ventas = pd.read_csv(f'{folder}/{name}_Ventas.csv')

    # Convertir tipos de datos    
    df_ventas['Codigo_Articulo'] = pd.to_numeric(df_ventas['Codigo_Articulo'], errors='coerce').astype('Int64')
    df_ventas['Sucursal'] = pd.to_numeric(df_ventas['Sucursal'], errors='coerce').astype('Int64')   
    df_ventas['Fecha']= pd.to_datetime(df_ventas['Fecha'])

    if df_ventas[['Codigo_Articulo', 'Sucursal']].isnull().any().any():
        print(f"⚠️ Atención: Ventas contiene registros con valores nulos en Código o Sucursal.")
        df_ventas = df_ventas.dropna(subset=['Codigo_Articulo', 'Sucursal'], how='all')  # Eliminar filas donde ambas columnas son NaN
        print(f"⚠️ Atención: Se ELIMINARON registros de Ventas que contiene registros con valores nulos en Código o Sucursal.")
    
    # Recuperar Maestro de Artículos
    articulos = pd.read_csv(f'{folder}/{name}_articulos.csv')
    
    # Recuperar Maestro de Artículos
    stock_sucursal = pd.read_csv(f'{folder}/{name}_stock_sucursal.csv')

    # Recuperando Forecast Calculado
    df_forecast = pd.read_csv(f'{folder}/{algoritmo}_Solicitudes_Compra.csv')
    df_forecast.fillna(0)   # Por si se filtró algún missing value
    print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {name}")
    
    conn = Open_Conn_Postgres()
    
    # Obtener Sites
    stores = pd.read_sql("SELECT code, name, id FROM public.fnd_site", conn) # type: ignore
    stores = stores[pd.to_numeric(stores['code'], errors='coerce').notna()].copy()
    stores['code'] = stores['code'].astype(int)

    # Obtener Productos
    products = pd.read_sql("SELECT ext_code, description, id FROM public.fnd_product", conn) # type: ignore
    products = products[pd.to_numeric(products['ext_code'], errors='coerce').notna()].copy()
    products['ext_code'] = products['ext_code'].astype(int)
    assert products['ext_code'].is_unique, "ERROR: productos.ext_code tiene duplicados"


    Close_Connection(conn)

    # Unir con productos y validar
    df_merged = df_forecast.merge(products, left_on='Codigo_Articulo', right_on='ext_code', how='left')
    df_merged.rename(columns={'id': 'product_id'}, inplace=True)
    df_merged.drop(columns=['ext_code', 'description'], inplace=True)

    # Unir con sites y validar
    df_merged = df_merged.merge(stores, left_on='Sucursal', right_on='code', how='left')
    df_merged.rename(columns={'id': 'site_id'}, inplace=True)
    df_merged.drop(columns=['code', 'name'], inplace=True)

    # Validación de integridad referencial
    errores = df_merged[df_merged['site_id'].isna() | df_merged['product_id'].isna()]
    if not errores.empty:
        print(f"❌ Error: Se encontraron {len(errores)} registros con site_id o product_id nulos.")
        errores[['Codigo_Articulo', 'Sucursal']].drop_duplicates().to_csv(
            f"{folder}/{algoritmo}_Errores_Missing_UUID.csv", index=False)
        raise ValueError("Existen artículos o sucursales no presentes en Connexa. Revisión necesaria.")

    
    df_nuevo = articulos.copy()   # Articulos ya tiene SP_BASE_ARTICULOS_SUCURSAL
    # -- COMBINAR ARTÍCULOS y STOCK --
    df_nuevo = pd.merge(df_nuevo, stock_sucursal, left_on=['C_ARTICULO', 'C_SUCU_EMPR'], right_on=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], how='inner')
    df_nuevo.drop(columns=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], inplace=True) 
    df_nuevo['C_SUCU_EMPR'] = df_nuevo['C_SUCU_EMPR'].astype(int)
    df_nuevo['C_ARTICULO'] = df_nuevo['C_ARTICULO'].astype(int)

    duplicados = df_nuevo[df_nuevo.duplicated(['C_SUCU_EMPR', 'C_ARTICULO'], keep=False)]
    if not duplicados.empty:
        print(f"🚨 Hay {len(duplicados)} filas duplicadas en df_nuevo por artículo+sucursal")
        duplicados.to_csv(f"{folder}/Duplicados_df_nuevo.csv", index=False)

        #Elimino Duplicados
        df_nuevo = df_nuevo.drop_duplicates(subset=['C_SUCU_EMPR', 'C_ARTICULO'])

    df_merged = df_merged.merge(
        df_nuevo,
        left_on=['Sucursal', 'Codigo_Articulo'],
        right_on=['C_SUCU_EMPR', 'C_ARTICULO'],
        how='left'
    )
    df_merged.drop(columns=['C_SUCU_EMPR', 'C_ARTICULO'], inplace=True)

    print(f"🔍 Forecast original: {len(df_forecast)} registros")
    print(f"📈 Forecast extendido: {len(df_merged)} registros")

    return df_merged

In [ ]:
name = "34229_Resistack"
algoritmo = "ALGO_05"
id_proveedor = 34229

In [ ]:
print(f'{folder}/{name}_{algoritmo}_Solicitudes_Compra.csv')

## PRUEBA INICIAL de DATAFRAMES


In [ ]:
# Recuperar Historial de Ventas
df_ventas = pd.read_csv(f'{folder}/{name}_Ventas.csv')

# Convertir tipos de datos    
df_ventas['Codigo_Articulo'] = pd.to_numeric(df_ventas['Codigo_Articulo'], errors='coerce').astype('Int64')
df_ventas['Sucursal'] = pd.to_numeric(df_ventas['Sucursal'], errors='coerce').astype('Int64')   
df_ventas['Fecha']= pd.to_datetime(df_ventas['Fecha'])

if df_ventas[['Codigo_Articulo', 'Sucursal']].isnull().any().any():
    print(f"⚠️ Atención: Ventas contiene registros con valores nulos en Código o Sucursal.")
    df_ventas = df_ventas.dropna(subset=['Codigo_Articulo', 'Sucursal'], how='all')  # Eliminar filas donde ambas columnas son NaN
    print(f"⚠️ Atención: Se ELIMINARON registros de Ventas que contiene registros con valores nulos en Código o Sucursal.")

In [ ]:
# Recuperar Maestro de Artículos
articulos = pd.read_csv(f'{folder}/{name}_articulos.csv')

# Recuperar Maestro de Artículos
stock_sucursal = pd.read_csv(f'{folder}/{name}_stock_sucursal.csv')

# Recuperando Forecast Calculado
df_forecast = pd.read_csv(f'{folder}/{name}_{algoritmo}_Solicitudes_Compra.csv')
df_forecast.fillna(0)   # Por si se filtró algún missing value
print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {name}")

In [ ]:
# Completar con UUIDs de Connexa
conn = Open_Conn_Postgres()
    
# Obtener Sites
stores = pd.read_sql("SELECT code, name, id FROM public.fnd_site", conn) # type: ignore
stores = stores[pd.to_numeric(stores['code'], errors='coerce').notna()].copy()
stores['code'] = stores['code'].astype(int)

# Obtener Productos
products = pd.read_sql("SELECT ext_code, description, id FROM public.fnd_product", conn) # type: ignore
products = products[pd.to_numeric(products['ext_code'], errors='coerce').notna()].copy()
products['ext_code'] = products['ext_code'].astype(int)
assert products['ext_code'].is_unique, "ERROR: productos.ext_code tiene duplicados"


Close_Connection(conn)

In [ ]:
# Unir con productos y validar
df_merged = df_forecast.merge(products, left_on='Codigo_Articulo', right_on='ext_code', how='left')
df_merged.rename(columns={'id': 'product_id'}, inplace=True)
df_merged.drop(columns=['ext_code', 'description'], inplace=True)


In [ ]:
# Unir con sites y validar
df_merged = df_merged.merge(stores, left_on='Sucursal', right_on='code', how='left')
df_merged.rename(columns={'id': 'site_id'}, inplace=True)
df_merged.drop(columns=['code', 'name'], inplace=True)

In [ ]:
# Validación de integridad referencial
errores = df_merged[df_merged['site_id'].isna() | df_merged['product_id'].isna()]
if not errores.empty:
    print(f"❌ Error: Se encontraron {len(errores)} registros con site_id o product_id nulos.")
    errores[['Codigo_Articulo', 'Sucursal']].drop_duplicates().to_csv(
        f"{folder}/{algoritmo}_Errores_Missing_UUID.csv", index=False)
    raise ValueError("Existen artículos o sucursales no presentes en Connexa. Revisión necesaria.")

In [ ]:
df_nuevo = articulos.copy()   # Articulos ya tiene SP_BASE_ARTICULOS_SUCURSAL
# -- COMBINAR ARTÍCULOS y STOCK --
df_nuevo = pd.merge(df_nuevo, stock_sucursal, left_on=['C_ARTICULO', 'C_SUCU_EMPR'], right_on=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], how='inner')
df_nuevo.drop(columns=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], inplace=True) 
df_nuevo['C_SUCU_EMPR'] = df_nuevo['C_SUCU_EMPR'].astype(int)
df_nuevo['C_ARTICULO'] = df_nuevo['C_ARTICULO'].astype(int)

In [ ]:
duplicados = df_nuevo[df_nuevo.duplicated(['C_SUCU_EMPR', 'C_ARTICULO'], keep=False)]
if not duplicados.empty:
    print(f"🚨 Hay {len(duplicados)} filas duplicadas en df_nuevo por artículo+sucursal")
    duplicados.to_csv(f"{folder}/Duplicados_df_nuevo.csv", index=False)

    #Elimino Duplicados
    df_nuevo = df_nuevo.drop_duplicates(subset=['C_SUCU_EMPR', 'C_ARTICULO'])

df_merged = df_merged.merge(
    df_nuevo,
    left_on=['Sucursal', 'Codigo_Articulo'],
    right_on=['C_SUCU_EMPR', 'C_ARTICULO'],
    how='left'
)
df_merged.drop(columns=['C_SUCU_EMPR', 'C_ARTICULO'], inplace=True)

print(f"🔍 Forecast original: {len(df_forecast)} registros")
print(f"📈 Forecast extendido: {len(df_merged)} registros")


# NUEVA VERSIÓN
### PRUEBA SELECCIÓN DE SURTIDO y SUCURSALES






In [4]:
# Funciones Utlilzadas

def Open_Connexa_Alquemy():
    DB_TYPE = "postgresql"
    DB_USER = secrets['PGP_USER']
    DB_PASS = secrets['PGP_PASSWORD']  # ⚠️ Reemplazar por la contraseña real o usar variables de entorno
    DB_HOST = secrets['PGP_HOST']
    DB_PORT = secrets['PGP_PORT']
    DB_NAME = secrets['PGP_DB']

    # Crear engine de conexión
    try:
        engine = sqlalchemy.create_engine(
        f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
        )
        conn = engine.connect()
        return conn
    except Exception as e:
        print(f'Error en la conexión: {e}')
        return None   
    
def Open_Diarco_Data_Alquemy():
    DB_TYPE = "postgresql"
    DB_USER = secrets['PG_USER']
    DB_PASS = secrets['PG_PASSWORD']  # ⚠️ Reemplazar por la contraseña real o usar variables de entorno
    DB_HOST = secrets['PG_HOST']
    DB_PORT = secrets['PG_PORT']
    DB_NAME = secrets['PG_DB']

    # Crear engine de conexión
    try:
        engine = sqlalchemy.create_engine(
        f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
        )
        conn = engine.connect()
        return conn
    except Exception as e:
        print(f'Error en la conexión: {e}')
        return None 
def get_forecast( id_proveedor, lbl_proveedor, period_lengh=30, algorithm='basic', f1=None, f2=None, f3=None, current_date=None, sucursales=None, rubros=None):
    """
    Genera la predicción de demanda según el algoritmo seleccionado.

    Parámetros:
    - id_proveedor: ID del proveedor.
    - lbl_proveedor: Etiqueta del proveedor.
    - period_lengh: Número de días del período a analizar (por defecto 30).
    - algorithm: Algoritmo a utilizar.
    - current_date: Fecha de referencia; si es None, se toma la fecha máxima de los datos.
    - factores de ponderación: F1, F2, F3  (No importa en que unidades estén, luego los hace relativos al total del peso)

    Retorna:
    - Un DataFrame con las predicciones.
    """
    
    print('Dentro del get_forecast LOCAL')
    print(f'FORECAST control: {id_proveedor} - {lbl_proveedor} - ventana: {period_lengh} - {algorithm} factores: {f1} - {f2} - {f3} ')
   
    # Generar los datos de entrada parampetricos
    if sucursales is not None:
        print(f'Filtrando por sucursales: {sucursales}')
        sucursales = ' IN (' + ','.join(map(str, sucursales)) + ')'
    else:
        sucursales = ' <> 0'
    if rubros is not None:
        print(f'Filtrando por rubros: {rubros}')
        rubros = ' IN (' + ','.join(f"'{r}'" for r in rubros) + ')'
    else:
        rubros = ' <> 0'
    
    data, articulos = generar_datos(id_proveedor, lbl_proveedor, period_lengh, sucursales, rubros) # type: ignore

    # Determinar la fecha base
    if current_date is None:
        current_date = data['Fecha'].max()  # type: ignore # Se toma la última fecha en los datos
    else:
        current_date = pd.to_datetime(current_date)  # Se asegura que sea un objeto datetime
    print(f'Fecha actual {current_date}')


def build_params(
    id_proveedor: int,
    desde: str | dt.date,
    sucursales: Optional[Iterable[int]] = None,
    rubros: Optional[Iterable[int]] = None
) -> Dict[str, Any]:
    """
    Devuelve el dict de parámetros en el formato esperado.
    Si sucursales o rubros vienen vacíos, los tratamos como NULL para no filtrar.
    """
    def _arr_or_null(x):
        if x is None:
            return None
        x = list(x)
        return x if len(x) > 0 else None

    print(f"[DEBUG] Parámetros construidos: id_proveedor={id_proveedor}, desde={desde}, sucursales={sucursales}, rubros={rubros}")
    return {
        "id_proveedor": int(id_proveedor),
        "desde": desde,                           # 'YYYY-MM-DD' o date
        "sucursales": _arr_or_null(sucursales),   # e.g. [1, 2, 3]  -> ANY(...)
        "rubros": _arr_or_null(rubros)            # e.g. [10, 20]   -> ANY(...)
    }

# Nueva Rutina al MIGRAR a PostgreSQL y Ejecución REMOTA
# 2025-05-16 Se agrega c_comprador
# 2025-06-21 Se agrega filtro por sucursales y rubros
def generar_datos(id_proveedor, etiqueta, ventana, sucursales, rubros):
    folder = secrets["FOLDER_DATOS"]
    archivo_datos = f'{folder}/{etiqueta}.csv'
    archivo_articulos = f'{folder}/{etiqueta}_articulos.csv'
    archivo_stock = f'{folder}/{etiqueta}_stock_sucursal.csv'
    
    # Verificar la fecha de modificación del archivo
    if os.path.exists(archivo_datos):
        fecha_modificacion = datetime.fromtimestamp(os.path.getmtime(archivo_datos))
        if fecha_modificacion.date() == datetime.today().date():
            try:
                data = pd.read_csv(archivo_datos)
                data = data.dropna(subset=['Codigo_Articulo', 'Sucursal', 'Fecha'], how='all')  # Eliminar filas donde todas las columnas clave son NaN
                data['Codigo_Articulo'] = data['Codigo_Articulo'].astype(int)
                data['Sucursal'] = data['Sucursal'].astype(int)
                data['Fecha'] = pd.to_datetime(data['Fecha'])

                articulos = pd.read_csv(archivo_articulos)
                articulos = articulos.dropna(subset=['C_ARTICULO', 'C_SUCU_EMPR'], how='all')  # Eliminar filas donde todas las columnas clave son NaN
                

                print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {etiqueta}")
                return data, articulos
            except Exception as e:
                print(f"Error al leer los archivos cacheados: {e}")
        else:
            # Eliminar archivos si la fecha no es la de hoy
            os.remove(archivo_datos)
            if os.path.exists(archivo_articulos):
                os.remove(archivo_articulos)
            print(f"-> Archivos eliminados por ser obsoletos: {archivo_datos}, {archivo_articulos}")
    else:
        print(f"-> Generando datos para ID: {id_proveedor}, Label: {etiqueta}")
        # Aquí puedes incluir el código para generar los datos si no EXISTE el Archivo en el CACHE
        conn = Open_Diarco_Data_Alquemy()

        # --- ARTÍCULOS --- NUEVA FUENTE GLOBAL 06/25 -- SP_BASE_PRODUCTOS_VIGENTES  (OJO NO FILTRA HABILITADOS)
        query_articulos = f"""
            SELECT DISTINCT P.c_sucu_empr, P.c_articulo, P.c_proveedor_primario, P.abastecimiento, P.cod_cd, P.habilitado,  
                    P.cod_comprador AS c_comprador, 
                    P.q_factor_compra, P.full_capacity_pallet, P.number_of_layers, P.number_of_boxes_per_layer
            FROM src.base_productos_vigentes P
            LEFT JOIN src.m_3_articulos A
            ON P.c_articulo = A.c_articulo
            WHERE P.c_proveedor_primario = {id_proveedor}
                AND P.c_sucu_empr {sucursales}   -- Filtrar por sucursales si se especifica
                AND A.c_rubro {rubros} -- Filtrar por rubros si se especifica
            AND habilitado = 1
            ORDER BY P.c_articulo, P.c_sucu_empr;
        """

        print(f"[DEBUG] Ejecutando query de ventas  {query_articulos}...")
        

        articulos = pd.read_sql(query_articulos, conn) # type: ignore
        if articulos.empty:
            print(f"❗ No se encontraron artículos para el proveedor {id_proveedor}.")
            Close_Connection(conn)
            return None, None

        articulos.columns = articulos.columns.str.upper()
        articulos.rename(columns={'COD_COMPRADOR': 'C_COMPRADOR'}, inplace=True)   # En la base nueva se llama distinto
        articulos['C_SUCU_EMPR'] = articulos['C_SUCU_EMPR'].astype(int)
        articulos['C_ARTICULO'] = articulos['C_ARTICULO'].astype(int)
        articulos['C_PROVEEDOR_PRIMARIO'] = articulos['C_PROVEEDOR_PRIMARIO'].astype(int)
        articulos['ABASTECIMIENTO'] = articulos['ABASTECIMIENTO'].astype(int)
        articulos['HABILITADO'] = articulos['HABILITADO'].astype(int)
        articulos['C_COMPRADOR'] = articulos['C_COMPRADOR'].astype(int)
        articulos['Q_FACTOR_COMPRA'] = articulos['Q_FACTOR_COMPRA'].astype(int)
        articulos['FULL_CAPACITY_PALLET'] = articulos['FULL_CAPACITY_PALLET'].astype(int)
        articulos['NUMBER_OF_LAYERS'] = articulos['NUMBER_OF_LAYERS'].astype(int)
        articulos['NUMBER_OF_BOXES_PER_LAYER'] = articulos['NUMBER_OF_BOXES_PER_LAYER'].astype(int)
        articulos.to_csv(archivo_articulos, index=False, encoding='utf-8')
        print(f"---> Datos de Artículos guardados")        
        
        #--- BASE STOCK --- NUEVA FUENTE GLOBAL 008/25 -- SP_BASE_STOCK_SUCURSAL
        query_stock_sucursal = f"""
            SELECT S.codigo_articulo, S.codigo_sucursal, S.codigo_proveedor, S.pedido_sgm, S.stock, 
                S.pedido_pendiente, S.i_lista_calculado, S.factor_venta, S.ultimo_ingreso, S.fecha_ultimo_ingreso,
                S.fecha_ultima_venta, S.precio_venta, S.precio_costo, 
                S.q_dias_stock, S.transfer_pendiente, S.venta_unidades_1q, S.venta_unidades_2q
            FROM src.base_stock_sucursal S
            LEFT JOIN src.m_3_articulos A
            ON S.codigo_articulo = A.c_articulo
            WHERE S.codigo_proveedor = {id_proveedor}
                AND S.codigo_sucursal {sucursales}   -- Filtrar por sucursales si se especifica
                AND A.c_rubro {rubros} -- Filtrar por rubros si se especifica
            ORDER BY S.codigo_articulo, S.codigo_articulo;
        """
        print(f"[DEBUG] Ejecutando query de stock_sucursal  {query_stock_sucursal}...")

        stock_sucursal = pd.read_sql(query_stock_sucursal, conn) # type: ignore
        if stock_sucursal.empty:
            print(f"❗ No se encontraron artículos de Stock_Sucursal para el proveedor {id_proveedor}.")
            Close_Connection(conn)
            return None, None
        
        # Limpieza general antes de conversión
        stock_sucursal = stock_sucursal.replace([np.inf, -np.inf], np.nan)
        stock_sucursal = stock_sucursal.fillna(0)

        stock_sucursal.columns = stock_sucursal.columns.str.upper()
        #  Cambiar tipos de datos
        stock_sucursal['CODIGO_SUCURSAL'] = stock_sucursal['CODIGO_SUCURSAL'].astype(int)
        stock_sucursal['CODIGO_ARTICULO'] = stock_sucursal['CODIGO_ARTICULO'].astype(int)
        stock_sucursal['CODIGO_PROVEEDOR'] = stock_sucursal['CODIGO_PROVEEDOR'].astype(int)
        stock_sucursal["PEDIDO_SGM"] = pd.to_numeric(stock_sucursal["PEDIDO_SGM"], errors="coerce").astype("Float64")
        stock_sucursal["STOCK"] = pd.to_numeric(stock_sucursal["STOCK"], errors="coerce").astype("Float64")
        stock_sucursal["PEDIDO_PENDIENTE"] = pd.to_numeric(stock_sucursal["PEDIDO_PENDIENTE"], errors="coerce").astype("Float64")
        stock_sucursal["I_LISTA_CALCULADO"] = pd.to_numeric(stock_sucursal["I_LISTA_CALCULADO"], errors="coerce").astype("Float64")
        stock_sucursal['FACTOR_VENTA'] = stock_sucursal['FACTOR_VENTA'].astype(int)
        stock_sucursal["ULTIMO_INGRESO"] = pd.to_numeric(stock_sucursal["ULTIMO_INGRESO"], errors="coerce").astype("Float64")
        stock_sucursal['PRECIO_VENTA'] = pd.to_numeric(stock_sucursal['PRECIO_VENTA'], errors='coerce').astype('Float64')
        stock_sucursal['PRECIO_COSTO'] = pd.to_numeric(stock_sucursal['PRECIO_COSTO'], errors='coerce').astype('Float64')
        stock_sucursal['Q_DIAS_STOCK'] = pd.to_numeric(stock_sucursal['Q_DIAS_STOCK'], errors='coerce').astype('Int64')
        stock_sucursal['TRANSFER_PENDIENTE'] = pd.to_numeric(stock_sucursal['TRANSFER_PENDIENTE'], errors='coerce').astype('Float64')
        stock_sucursal['VENTA_UNIDADES_1Q'] = pd.to_numeric(stock_sucursal['VENTA_UNIDADES_1Q'], errors='coerce').astype('Float64')
        stock_sucursal['VENTA_UNIDADES_2Q'] = pd.to_numeric(stock_sucursal['VENTA_UNIDADES_2Q'], errors='coerce').astype('Float64')

        stock_sucursal.to_csv(archivo_stock, index=False, encoding='utf-8')
        print(f"---> Datos de Stock Sucursal guardados")
        
        # -- COMBINAR ARTÍCULOS y STOCK --
        articulos = pd.merge(articulos, stock_sucursal, left_on=['C_ARTICULO', 'C_SUCU_EMPR'], right_on=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], how='inner')  

        # --- VENTAS --- DIARCO + BARRIO ( En 2 Bases de Datos distintas )
        query_ventas = """
            WITH ventas_unificadas AS (
                SELECT v.f_venta        AS fecha,
                    v.c_articulo     AS codigo_articulo,
                    v.c_sucu_empr    AS sucursal,
                    v.q_unidades_vendidas AS unidades
                FROM src.t702_est_vtas_por_articulo v
                WHERE v.f_venta >= %(desde)s

                UNION ALL

                SELECT v.f_venta        AS fecha,
                    v.c_articulo     AS codigo_articulo,
                    v.c_sucu_empr    AS sucursal,
                    v.q_unidades_vendidas AS unidades
                FROM src.t702_est_vtas_por_articulo_dbarrio v
                WHERE v.f_venta >= %(desde)s
            ),
            articulos_filtrados AS (
                /* Filtra artículos por rubro solo si se especifican rubros */
                SELECT m.c_articulo
                FROM src.m_3_articulos m
                WHERE %(rubros)s IS NULL
                OR m.c_rubro = ANY (%(rubros)s)
            ),
            surtido_filtrado AS (
                /* Surtido habilitado por proveedor (y opcionalmente por sucursal) */
                SELECT a.c_articulo  AS codigo_articulo,
                    a.c_sucu_empr AS sucursal
                FROM src.base_productos_vigentes a
                /* Proveedor siempre obligatorio */
                WHERE a.c_proveedor_primario = %(id_proveedor)s
                /* Sucursales opcionales */
                AND (%(sucursales)s IS NULL OR a.c_sucu_empr = ANY (%(sucursales)s))
                /* Si hay filtro de rubros, intersectamos con los artículos filtrados */
                AND (%(rubros)s IS NULL OR a.c_articulo IN (SELECT c_articulo FROM articulos_filtrados))
            )
            SELECT v.fecha,
                v.codigo_articulo,
                v.sucursal,
                v.unidades
            FROM ventas_unificadas v
            JOIN surtido_filtrado s
            ON s.codigo_articulo = v.codigo_articulo
            AND s.sucursal        = v.sucursal
            ORDER BY v.fecha, v.codigo_articulo, v.sucursal;
        """

        params = build_params(
        id_proveedor=id_proveedor,
        desde = "2024-06-01",
        sucursales=sucursales,    # o None si no quieren filtrar
        rubros=rubros         # o [101, 102]
        )

    
        # ur.execute(query_ventas, params)   # psycopg2
       
        ventas = conn.execute(text(query_ventas), params)   # o con SQLAlchemy:

        print(f"[DEBUG] Ejecutando query de ventas  {query_ventas}...")
        print(f"[DEBUG] Parámetros: {params}")

        ventas = pd.read_sql(query_ventas, conn) # type: ignore
        if ventas.empty:
            print(f"⚠️ No se encontraron ventas DIARCO + BARRIO para el proveedor {id_proveedor}.")
        
        # Convertir columnas a minúsculas si hay datos
        if not ventas.empty:
            ventas.columns = ventas.columns.str.lower()
            # Transformar tipos de datos si hay datos
            ventas['sucursal'] = ventas['sucursal'].astype(int)
            ventas['codigo_articulo'] = ventas['codigo_articulo'].astype(int)
            ventas['fecha'] = pd.to_datetime(ventas['fecha'])
            # Eliminar filas con NaN en 'fecha', 'codigo_articulo' o 'sucursal'
            ventas = ventas.dropna(subset=['fecha', 'codigo_articulo', 'sucursal'], how='all')
            
        else:
            print(f"⚠️ No se encontraron ventas para el proveedor {id_proveedor} ni en DIARCO ni en BARRIO.")
            ventas = pd.DataFrame(columns=['fecha', 'codigo_articulo', 'sucursal', 'unidades'])  # DataFrame vacío con columnas esperadas

        ventas = ventas.rename(columns={
            "fecha": "Fecha",
            "codigo_articulo": "Codigo_Articulo",
            "sucursal": "Sucursal",
            "unidades": "Unidades"
        })
        # Guardar solo VENTAS 
        ventas.to_csv(f'{folder}/{etiqueta}_Demanda.csv', index=False, encoding='utf-8')
        print(f"---> Datos de Ventas guardados")

        # --- MERGE ---
        data = pd.merge(
            articulos,
            ventas,  
            left_on=['C_ARTICULO', 'C_SUCU_EMPR'],          
            right_on=['Codigo_Articulo', 'Sucursal'],            
            how='left'   # 'inner'  # Solo traer los productos que están en AMBOS ARCHIVOS  'left' trate TODOS los articulos activos en cada sucursal
        )

        if data.empty:
            print(f"⚠️ No hay coincidencias entre artículos y ventas para el proveedor {id_proveedor}.")
            Close_Connection(conn)
            return None, articulos

        # Conversión segura de columnas a enteros con soporte para NaN
        data['C_ARTICULO'] = data['C_ARTICULO'].astype('Int64')
        data['C_SUCU_EMPR'] = data['C_SUCU_EMPR'].astype('Int64')
        data['Codigo_Articulo'] = data['Codigo_Articulo'].astype('Int64')
        data['Sucursal'] = data['Sucursal'].astype('Int64')

        # Agregar columna indicadora de si tiene demanda
        data['TIENE_DEMANDA'] = data['Unidades'].notna().astype(int)
        data['Unidades'] = data['Unidades'].fillna(0)

        # Guardado
        data.to_csv(archivo_datos, index=False, encoding='utf-8')
        print(f"---> Datos de RECUPERACIÓN guardados")
        
        # Guardar los datos Compactos de VENTAS en un archivo CSV con el nombre del Proveedor y sufijo _Ventas
        file_path = f'{folder}/{etiqueta}_Ventas.csv'
        print(f"[DEBUG] Ruta destino definida en .env: {folder}")

        # Eliminar Columnas Innecesarias
        data = data[['Fecha', 'Codigo_Articulo', 'Sucursal', 'Unidades']]
        data.to_csv(file_path, index=False, encoding='utf-8')
        print(f"---> Datos de Ventas guardados: {file_path}")  

        Close_Connection(conn)
        return data, articulos


In [ ]:
from typing import Iterable, Optional, Tuple, Dict, Any
import datetime as dt


In [6]:
name = "20_ARCOR"
algoritmo = "ALGO_05"
id_proveedor = 20
sucursales =  [1, 2, 49,302] # Ejemplo de sucursales a filtrar
rubros = [2055, 2053]  # Ejemplo de rubros a filtrar


get_forecast( id_proveedor, name, period_lengh=90, algorithm=algoritmo, f1=0.5, f2=0.3, f3=0.2, current_date=None, sucursales=sucursales, rubros=rubros)  


Dentro del get_forecast LOCAL
FORECAST control: 20 - 20_ARCOR - ventana: 90 - ALGO_05 factores: 0.5 - 0.3 - 0.2 
Filtrando por sucursales: [1, 2, 49, 302]
Filtrando por rubros: [2055, 2053]
-> Generando datos para ID: 20, Label: 20_ARCOR
[DEBUG] Ejecutando query de ventas  
            SELECT DISTINCT P.c_sucu_empr, P.c_articulo, P.c_proveedor_primario, P.abastecimiento, P.cod_cd, P.habilitado,  
                    P.cod_comprador AS c_comprador, 
                    P.q_factor_compra, P.full_capacity_pallet, P.number_of_layers, P.number_of_boxes_per_layer
            FROM src.base_productos_vigentes P
            LEFT JOIN src.m_3_articulos A
            ON P.c_articulo = A.c_articulo
            WHERE P.c_proveedor_primario = 20
                AND P.c_sucu_empr  IN (1,2,49,302)   -- Filtrar por sucursales si se especifica
                AND A.c_rubro  IN ('2055','2053') -- Filtrar por rubros si se especifica
            AND habilitado = 1
            ORDER BY P.c_articulo, P.c_suc

OSError: Cannot save file into a non-existent directory: 'data'

In [ ]:
# Nueva Rutina al MIGRAR a PostgreSQL y Ejecución REMOTA
# 2025-05-16 Se agrega c_comprador
# 2025-06-21 Se agrega filtro por sucursales y rubros
def generar_datos(id_proveedor, etiqueta, ventana, sucursales, rubros):
    folder = secrets["FOLDER_DATOS"]
    archivo_datos = f'{folder}/{etiqueta}.csv'
    archivo_articulos = f'{folder}/{etiqueta}_articulos.csv'
    archivo_stock = f'{folder}/{etiqueta}_stock_sucursal.csv'
    
    # Verificar la fecha de modificación del archivo
    if os.path.exists(archivo_datos):
        fecha_modificacion = datetime.fromtimestamp(os.path.getmtime(archivo_datos))
        if fecha_modificacion.date() == datetime.today().date():
            try:
                data = pd.read_csv(archivo_datos)
                data = data.dropna(subset=['Codigo_Articulo', 'Sucursal', 'Fecha'], how='all')  # Eliminar filas donde todas las columnas clave son NaN
                data['Codigo_Articulo'] = data['Codigo_Articulo'].astype(int)
                data['Sucursal'] = data['Sucursal'].astype(int)
                data['Fecha'] = pd.to_datetime(data['Fecha'])

                articulos = pd.read_csv(archivo_articulos)
                articulos = articulos.dropna(subset=['C_ARTICULO', 'C_SUCU_EMPR'], how='all')  # Eliminar filas donde todas las columnas clave son NaN
                

                print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {etiqueta}")
                return data, articulos
            except Exception as e:
                print(f"Error al leer los archivos cacheados: {e}")
        else:
            # Eliminar archivos si la fecha no es la de hoy
            os.remove(archivo_datos)
            if os.path.exists(archivo_articulos):
                os.remove(archivo_articulos)
            print(f"-> Archivos eliminados por ser obsoletos: {archivo_datos}, {archivo_articulos}")
    else:
        print(f"-> Generando datos para ID: {id_proveedor}, Label: {etiqueta}")
        # Aquí puedes incluir el código para generar los datos si no EXISTE el Archivo en el CACHE
        conn = Open_Connexa_Alquemy()

        # --- ARTÍCULOS --- NUEVA FUENTE GLOBAL 06/25 -- SP_BASE_PRODUCTOS_VIGENTES  (OJO NO FILTRA HABILITADOS)
        query_articulos = f"""
            SELECT DISTINCT P.c_sucu_empr, P.c_articulo, P.c_proveedor_primario, P.abastecimiento, P.cod_cd, P.habilitado,  
                    P.cod_comprador AS c_comprador, 
                    P.q_factor_compra, P.full_capacity_pallet, P.number_of_layers, P.number_of_boxes_per_layer
            FROM src.base_productos_vigentes P
            LEFT JOIN src.m_3_articulos A
            ON P.c_articulo = A.c_articulo
            WHERE P.c_proveedor_primario = {id_proveedor}
                AND P.c_sucu_empr {sucursales}   -- Filtrar por sucursales si se especifica
                AND A.c_rubro {rubros} -- Filtrar por rubros si se especifica
            AND habilitado = 1
            ORDER BY P.c_articulo, P.c_sucu_empr;
        """

        print(f"[DEBUG] Ejecutando query de ventas  {query_articulos}...")
        

        # articulos = pd.read_sql(query_articulos, conn) # type: ignore
        # if articulos.empty:
        #     print(f"❗ No se encontraron artículos para el proveedor {id_proveedor}.")
        #     Close_Connection(conn)
        #     return None, None

        # articulos.columns = articulos.columns.str.upper()
        # articulos.rename(columns={'COD_COMPRADOR': 'C_COMPRADOR'}, inplace=True)   # En la base nueva se llama distinto
        # articulos['C_SUCU_EMPR'] = articulos['C_SUCU_EMPR'].astype(int)
        # articulos['C_ARTICULO'] = articulos['C_ARTICULO'].astype(int)
        # articulos['C_PROVEEDOR_PRIMARIO'] = articulos['C_PROVEEDOR_PRIMARIO'].astype(int)
        # articulos['ABASTECIMIENTO'] = articulos['ABASTECIMIENTO'].astype(int)
        # articulos['HABILITADO'] = articulos['HABILITADO'].astype(int)
        # articulos['C_COMPRADOR'] = articulos['C_COMPRADOR'].astype(int)
        # articulos['Q_FACTOR_COMPRA'] = articulos['Q_FACTOR_COMPRA'].astype(int)
        # articulos['FULL_CAPACITY_PALLET'] = articulos['FULL_CAPACITY_PALLET'].astype(int)
        # articulos['NUMBER_OF_LAYERS'] = articulos['NUMBER_OF_LAYERS'].astype(int)
        # articulos['NUMBER_OF_BOXES_PER_LAYER'] = articulos['NUMBER_OF_BOXES_PER_LAYER'].astype(int)
        # articulos.to_csv(archivo_articulos, index=False, encoding='utf-8')
        # print(f"---> Datos de Artículos guardados")        
        
        # --- BASE STOCK --- NUEVA FUENTE GLOBAL 06/25 -- SP_BASE_STOCK_SUCURSAL
        query_stock_sucursal = f"""
            SELECT S.codigo_articulo, S.codigo_sucursal, S.codigo_proveedor, S.pedido_sgm, S.stock, 
                S.pedido_pendiente, S.i_lista_calculado, S.factor_venta, S.ultimo_ingreso, S.fecha_ultimo_ingreso,
                S.fecha_ultima_venta, S.precio_venta, S.precio_costo, 
                S.q_dias_stock, S.transfer_pendiente, S.venta_unidades_1q, S.venta_unidades_2q
            FROM src.base_stock_sucursal S
            LEFT JOIN src.m_3_articulos A
            ON S.codigo_articulo = A.c_articulo
            WHERE S.codigo_proveedor = {id_proveedor}
                AND S.codigo_sucursal {sucursales}   -- Filtrar por sucursales si se especifica
                AND A.c_rubro {rubros} -- Filtrar por rubros si se especifica
            ORDER BY S.codigo_articulo, S.codigo_articulo;
        """
        print(f"[DEBUG] Ejecutando query de stock_sucursal  {query_stock_sucursal}...")

        # stock_sucursal = pd.read_sql(query_stock_sucursal, conn) # type: ignore
        # if stock_sucursal.empty:
        #     print(f"❗ No se encontraron artículos de Stock_Sucursal para el proveedor {id_proveedor}.")
        #     Close_Connection(conn)
        #     return None, None
        
        # # Limpieza general antes de conversión
        # stock_sucursal = stock_sucursal.replace([np.inf, -np.inf], np.nan)
        # stock_sucursal = stock_sucursal.fillna(0)

        # stock_sucursal.columns = stock_sucursal.columns.str.upper()
        # #  Cambiar tipos de datos
        # stock_sucursal['CODIGO_SUCURSAL'] = stock_sucursal['CODIGO_SUCURSAL'].astype(int)
        # stock_sucursal['CODIGO_ARTICULO'] = stock_sucursal['CODIGO_ARTICULO'].astype(int)
        # stock_sucursal['CODIGO_PROVEEDOR'] = stock_sucursal['CODIGO_PROVEEDOR'].astype(int)
        # stock_sucursal["PEDIDO_SGM"] = pd.to_numeric(stock_sucursal["PEDIDO_SGM"], errors="coerce").astype("Float64")
        # stock_sucursal["STOCK"] = pd.to_numeric(stock_sucursal["STOCK"], errors="coerce").astype("Float64")
        # stock_sucursal["PEDIDO_PENDIENTE"] = pd.to_numeric(stock_sucursal["PEDIDO_PENDIENTE"], errors="coerce").astype("Float64")
        # stock_sucursal["I_LISTA_CALCULADO"] = pd.to_numeric(stock_sucursal["I_LISTA_CALCULADO"], errors="coerce").astype("Float64")
        # stock_sucursal['FACTOR_VENTA'] = stock_sucursal['FACTOR_VENTA'].astype(int)
        # stock_sucursal["ULTIMO_INGRESO"] = pd.to_numeric(stock_sucursal["ULTIMO_INGRESO"], errors="coerce").astype("Float64")
        # stock_sucursal['PRECIO_VENTA'] = pd.to_numeric(stock_sucursal['PRECIO_VENTA'], errors='coerce').astype('Float64')
        # stock_sucursal['PRECIO_COSTO'] = pd.to_numeric(stock_sucursal['PRECIO_COSTO'], errors='coerce').astype('Float64')
        # stock_sucursal['Q_DIAS_STOCK'] = pd.to_numeric(stock_sucursal['Q_DIAS_STOCK'], errors='coerce').astype('Int64')
        # stock_sucursal['TRANSFER_PENDIENTE'] = pd.to_numeric(stock_sucursal['TRANSFER_PENDIENTE'], errors='coerce').astype('Float64')
        # stock_sucursal['VENTA_UNIDADES_1Q'] = pd.to_numeric(stock_sucursal['VENTA_UNIDADES_1Q'], errors='coerce').astype('Float64')
        # stock_sucursal['VENTA_UNIDADES_2Q'] = pd.to_numeric(stock_sucursal['VENTA_UNIDADES_2Q'], errors='coerce').astype('Float64')

        # stock_sucursal.to_csv(archivo_stock, index=False, encoding='utf-8')
        # print(f"---> Datos de Stock Sucursal guardados")
        
        # # -- COMBINAR ARTÍCULOS y STOCK --
        # articulos = pd.merge(articulos, stock_sucursal, left_on=['C_ARTICULO', 'C_SUCU_EMPR'], right_on=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], how='inner')  

        # --- VENTAS --- DIARCO + BARRIO ( En 2 Bases de Datos distintas )
        query_ventas = """
            WITH ventas_unificadas AS (
                SELECT v.f_venta        AS fecha,
                    v.c_articulo     AS codigo_articulo,
                    v.c_sucu_empr    AS sucursal,
                    v.q_unidades_vendidas AS unidades
                FROM src.t702_est_vtas_por_articulo v
                WHERE v.f_venta >= %(desde)s

                UNION ALL

                SELECT v.f_venta        AS fecha,
                    v.c_articulo     AS codigo_articulo,
                    v.c_sucu_empr    AS sucursal,
                    v.q_unidades_vendidas AS unidades
                FROM src.t702_est_vtas_por_articulo_dbarrio v
                WHERE v.f_venta >= %(desde)s
            ),
            articulos_filtrados AS (
                /* Filtra artículos por rubro solo si se especifican rubros */
                SELECT m.c_articulo
                FROM src.m_3_articulos m
                WHERE %(rubros)s IS NULL
                OR m.c_rubro = ANY (%(rubros)s)
            ),
            surtido_filtrado AS (
                /* Surtido habilitado por proveedor (y opcionalmente por sucursal) */
                SELECT a.c_articulo  AS codigo_articulo,
                    a.c_sucu_empr AS sucursal
                FROM src.base_productos_vigentes a
                /* Proveedor siempre obligatorio */
                WHERE a.c_proveedor_primario = %(id_proveedor)s
                /* Sucursales opcionales */
                AND (%(sucursales)s IS NULL OR a.c_sucu_empr = ANY (%(sucursales)s))
                /* Si hay filtro de rubros, intersectamos con los artículos filtrados */
                AND (%(rubros)s IS NULL OR a.c_articulo IN (SELECT c_articulo FROM articulos_filtrados))
            )
            SELECT v.fecha,
                v.codigo_articulo,
                v.sucursal,
                v.unidades
            FROM ventas_unificadas v
            JOIN surtido_filtrado s
            ON s.codigo_articulo = v.codigo_articulo
            AND s.sucursal        = v.sucursal
            ORDER BY v.fecha, v.codigo_articulo, v.sucursal;
        """

        params = build_params(
        id_proveedor=id_proveedor,
        desde = "2024-06-01",
        sucursales=sucursales,    # o None si no quieren filtrar
        rubros=rubros         # o [101, 102]
        )

    
        #cur.execute(query_ventas, params)   # psycopg2
       
        #ventas = conn.execute(text(query_ventas), params)   # o con SQLAlchemy:

        print(f"[DEBUG] Ejecutando query de ventas  {query_ventas}...")
        print(f"[DEBUG] Parámetros: {params}")

        # ventas = pd.read_sql(query_ventas, conn) # type: ignore
        # if ventas.empty:
        #     print(f"⚠️ No se encontraron ventas DIARCO + BARRIO para el proveedor {id_proveedor}.")
        
        # # Convertir columnas a minúsculas si hay datos
        # if not ventas.empty:
        #     ventas.columns = ventas.columns.str.lower()
        #     # Transformar tipos de datos si hay datos
        #     ventas['sucursal'] = ventas['sucursal'].astype(int)
        #     ventas['codigo_articulo'] = ventas['codigo_articulo'].astype(int)
        #     ventas['fecha'] = pd.to_datetime(ventas['fecha'])
        #     # Eliminar filas con NaN en 'fecha', 'codigo_articulo' o 'sucursal'
        #     ventas = ventas.dropna(subset=['fecha', 'codigo_articulo', 'sucursal'], how='all')
            
        # else:
        #     print(f"⚠️ No se encontraron ventas para el proveedor {id_proveedor} ni en DIARCO ni en BARRIO.")
        #     ventas = pd.DataFrame(columns=['fecha', 'codigo_articulo', 'sucursal', 'unidades'])  # DataFrame vacío con columnas esperadas

        # ventas = ventas.rename(columns={
        #     "fecha": "Fecha",
        #     "codigo_articulo": "Codigo_Articulo",
        #     "sucursal": "Sucursal",
        #     "unidades": "Unidades"
        # })
        # # Guardar solo VENTAS 
        # ventas.to_csv(f'{folder}/{etiqueta}_Demanda.csv', index=False, encoding='utf-8')
        # print(f"---> Datos de Ventas guardados")

        # # --- MERGE ---
        # data = pd.merge(
        #     articulos,
        #     ventas,  
        #     left_on=['C_ARTICULO', 'C_SUCU_EMPR'],          
        #     right_on=['Codigo_Articulo', 'Sucursal'],            
        #     how='left'   # 'inner'  # Solo traer los productos que están en AMBOS ARCHIVOS  'left' trate TODOS los articulos activos en cada sucursal
        # )

        # if data.empty:
        #     print(f"⚠️ No hay coincidencias entre artículos y ventas para el proveedor {id_proveedor}.")
        #     Close_Connection(conn)
        #     return None, articulos

        # # Conversión segura de columnas a enteros con soporte para NaN
        # data['C_ARTICULO'] = data['C_ARTICULO'].astype('Int64')
        # data['C_SUCU_EMPR'] = data['C_SUCU_EMPR'].astype('Int64')
        # data['Codigo_Articulo'] = data['Codigo_Articulo'].astype('Int64')
        # data['Sucursal'] = data['Sucursal'].astype('Int64')

        # # Agregar columna indicadora de si tiene demanda
        # data['TIENE_DEMANDA'] = data['Unidades'].notna().astype(int)
        # data['Unidades'] = data['Unidades'].fillna(0)

        # # Guardado
        # data.to_csv(archivo_datos, index=False, encoding='utf-8')
        # print(f"---> Datos de RECUPERACIÓN guardados")
        
        # # Guardar los datos Compactos de VENTAS en un archivo CSV con el nombre del Proveedor y sufijo _Ventas
        # file_path = f'{folder}/{etiqueta}_Ventas.csv'
        # print(f"[DEBUG] Ruta destino definida en .env: {folder}")

        # # Eliminar Columnas Innecesarias
        # data = data[['Fecha', 'Codigo_Articulo', 'Sucursal', 'Unidades']]
        # data.to_csv(file_path, index=False, encoding='utf-8')
        # print(f"---> Datos de Ventas guardados: {file_path}")  

        # Close_Connection(conn)
        return data, articulos


# VERSIÓN FINAL

In [ ]:
import os
import sqlalchemy
from sqlalchemy import text, bindparam
from sqlalchemy.engine import Connection
from sqlalchemy.dialects.postgresql import ARRAY, INTEGER
from contextlib import contextmanager

# Asumimos que 'secrets' ya existe con las claves
def Open_Connexa_Alquemy() -> Connection:
    DB_TYPE = "postgresql+psycopg2"
    DB_USER = secrets['PGP_USER']
    DB_PASS = secrets['PGP_PASSWORD']
    DB_HOST = secrets['PGP_HOST']
    DB_PORT = secrets['PGP_PORT']
    DB_NAME = secrets['PGP_DB']

    print(f"[DEBUG] Conectando a PostgreSQL en {DB_HOST}:{DB_PORT}, Base de Datos: {DB_NAME}...")

    engine = sqlalchemy.create_engine(
        f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
        pool_pre_ping=True,               # evita conexiones muertas
        pool_size=5, max_overflow=10,     # tunning básico
        future=True
    )
    return engine.connect()

# print("Conexión a PostgreSQL con SQLAlchemy establecida.")
# print(f"[DEBUG] Host: {secrets['PGP_HOST']}, Port: {secrets['PGP_PORT']}, DB: {secrets['PGP_DB']}" )

# conn_str = f"dbname={secrets['PGP_DB']} user={secrets['PGP_USER']} password={secrets['PGP_PASSWORD']} host={secrets['PGP_HOST']} port={secrets['PGP_PORT']}"

from typing import Iterable, Optional, Dict, Any
import datetime as dt


def Open_Diarco_Data_Alquemy() -> Connection:
    DB_TYPE = "postgresql+psycopg2"
    DB_USER = secrets['PG_USER']
    DB_PASS = secrets['PG_PASSWORD']
    DB_HOST = secrets['PG_HOST']
    DB_PORT = secrets['PG_PORT']
    DB_NAME = secrets['PG_DB']

    print(f"[DEBUG] Conectando a PostgreSQL en {DB_HOST}:{DB_PORT}, Base de Datos: {DB_NAME}...")

    engine = sqlalchemy.create_engine(
        f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
        pool_pre_ping=True,               # evita conexiones muertas
        pool_size=5, max_overflow=10,     # tunning básico
        future=True
    )
    return engine.connect()

print("Conexión a PostgreSQL con SQLAlchemy establecida.")
print(f"[DEBUG] Host: {secrets['PG_HOST']}, Port: {secrets['PG_PORT']}, DB: {secrets['PG_DB']}" )

conn_str = f"dbname={secrets['PG_DB']} user={secrets['PG_USER']} password={secrets['PG_PASSWORD']} host={secrets['PG_HOST']} port={secrets['PG_PORT']}"

from typing import Iterable, Optional, Dict, Any
import datetime as dt

def build_params(
    id_proveedor: int,
    desde: str | dt.date,
    sucursales: Optional[Iterable[int]] = None,
    rubros: Optional[Iterable[int]] = None
) -> Dict[str, Any]:
    def _arr_or_null(x):
        if x is None:
            return None
        x = list(x)
        return x if len(x) > 0 else None

    print(f"[DEBUG] Parámetros construidos: id_proveedor={id_proveedor}, desde={desde}, sucursales={sucursales}, rubros={rubros}")
    return {
        "id_proveedor": int(id_proveedor),
        "desde": desde,
        "sucursales": _arr_or_null(sucursales),  # p.ej. [1,2,3] o None
        "rubros": _arr_or_null(rubros)           # p.ej. [2055,2053] o None
    }

def get_forecast(id_proveedor, lbl_proveedor, period_lengh=30, algorithm='basic',
                 f1=None, f2=None, f3=None, current_date=None,
                 sucursales=None, rubros=None):
    """
    Genera la predicción de demanda según el algoritmo seleccionado.
    """
    print('Dentro del get_forecast LOCAL')
    print(f'FORECAST control: {id_proveedor} - {lbl_proveedor} - ventana: {period_lengh} - {algorithm} factores: {f1} - {f2} - {f3} ')
    if sucursales is not None:
        print(f'Filtrando por sucursales (array): {sucursales}')
    if rubros is not None:
        print(f'Filtrando por rubros (array): {rubros}')

    data, articulos = generar_datos(id_proveedor, lbl_proveedor, period_lengh, sucursales, rubros)  # ojo: ahora pasa arrays

    # Determinar la fecha base
    if data is not None and not data.empty:
        if current_date is None:
            current_date = data['Fecha'].max()
        else:
            current_date = pd.to_datetime(current_date)
        print(f'Fecha actual {current_date}')
    else:
        print("⚠️ No hay datos para determinar current_date.")
    return data, articulos

from sqlalchemy import text, bindparam
from sqlalchemy.dialects.postgresql import ARRAY, INTEGER
import pandas as pd
import numpy as np
import os
from datetime import datetime

def generar_datos(id_proveedor, etiqueta, ventana, sucursales, rubros):
    folder = secrets["FOLDER_DATOS"]
    archivo_datos = f'{folder}/{etiqueta}.csv'
    archivo_articulos = f'{folder}/{etiqueta}_articulos.csv'
    archivo_stock = f'{folder}/{etiqueta}_stock_sucursal.csv'

    # ---------- CACHE ----------
    if os.path.exists(archivo_datos):
        fecha_modificacion = datetime.fromtimestamp(os.path.getmtime(archivo_datos))
        if fecha_modificacion.date() == datetime.today().date():
            try:
                data = pd.read_csv(archivo_datos)
                data = data.dropna(subset=['Codigo_Articulo', 'Sucursal', 'Fecha'], how='all')
                data['Codigo_Articulo'] = pd.to_numeric(data['Codigo_Articulo'], errors='coerce').astype('Int64')
                data['Sucursal'] = pd.to_numeric(data['Sucursal'], errors='coerce').astype('Int64')
                data['Fecha'] = pd.to_datetime(data['Fecha'])

                articulos = pd.read_csv(archivo_articulos)
                articulos = articulos.dropna(subset=['C_ARTICULO', 'C_SUCU_EMPR'], how='all')
                print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {etiqueta}")
                return data, articulos
            except Exception as e:
                print(f"Error al leer los archivos cacheados: {e}")
        else:
            # limpiar cache viejo
            try:
                os.remove(archivo_datos)
            except FileNotFoundError:
                pass
            if os.path.exists(archivo_articulos):
                try:
                    os.remove(archivo_articulos)
                except FileNotFoundError:
                    pass
            print(f"-> Archivos eliminados por ser obsoletos: {archivo_datos}, {archivo_articulos}")

    print(f"-> Generando datos para ID: {id_proveedor}, Label: {etiqueta}")

    # Armado de parámetros comunes (arrays o NULL)
    sucursales = list(sucursales) if sucursales else None
    rubros     = list(rubros) if rubros else None

    with Open_Diarco_Data_Alquemy() as conn:
        # ---------------------------
        # 4.1) ARTÍCULOS (parametrizado)
        # ---------------------------
        sql_articulos = text("""
            SELECT DISTINCT P.c_sucu_empr, P.c_articulo, P.c_proveedor_primario, P.abastecimiento, P.cod_cd, P.habilitado,
                   P.cod_comprador AS c_comprador,
                   P.q_factor_compra, P.full_capacity_pallet, P.number_of_layers, P.number_of_boxes_per_layer
            FROM src.base_productos_vigentes P
            LEFT JOIN src.m_3_articulos A
                   ON P.c_articulo = A.c_articulo
            WHERE P.c_proveedor_primario = :id_proveedor
              AND (:sucursales IS NULL OR P.c_sucu_empr = ANY(:sucursales))
              AND (:rubros     IS NULL OR A.c_rubro     = ANY(:rubros))
              AND P.habilitado = 1
            ORDER BY P.c_articulo, P.c_sucu_empr
        """).bindparams(
            bindparam("id_proveedor", type_=INTEGER),
            bindparam("sucursales",   type_=ARRAY(INTEGER)),
            bindparam("rubros",       type_=ARRAY(INTEGER)),
        )

        print("[DEBUG] Ejecutando query de artículos...")
        articulos = pd.read_sql(
            sql_articulos, conn,
            params={"id_proveedor": id_proveedor, "sucursales": sucursales, "rubros": rubros}
        )

        if articulos.empty:
            print(f"❗ No se encontraron artículos para el proveedor {id_proveedor}.")
            return None, None

        articulos.columns = articulos.columns.str.upper()
        articulos.rename(columns={'COD_COMPRADOR': 'C_COMPRADOR'}, inplace=True)

        # Cast consistente a tipos enteros “nullable”
        for c in ['C_SUCU_EMPR','C_ARTICULO','C_PROVEEDOR_PRIMARIO','ABASTECIMIENTO','HABILITADO','C_COMPRADOR',
                  'Q_FACTOR_COMPRA','FULL_CAPACITY_PALLET','NUMBER_OF_LAYERS','NUMBER_OF_BOXES_PER_LAYER']:
            articulos[c] = pd.to_numeric(articulos[c], errors='coerce').astype('Int64')

        articulos.to_csv(archivo_articulos, index=False, encoding='utf-8')
        print("---> Datos de Artículos guardados")

        # -------------------------------------
        # 4.2) STOCK POR SUCURSAL (parametrizado)
        # -------------------------------------
        sql_stock = text("""
            SELECT S.codigo_articulo, S.codigo_sucursal, S.codigo_proveedor, S.pedido_sgm, S.stock,
                   S.pedido_pendiente, S.i_lista_calculado, S.factor_venta, S.ultimo_ingreso, S.fecha_ultimo_ingreso,
                   S.fecha_ultima_venta, S.precio_venta, S.precio_costo,
                   S.q_dias_stock, S.transfer_pendiente, S.venta_unidades_1q, S.venta_unidades_2q
            FROM src.base_stock_sucursal S
            LEFT JOIN src.m_3_articulos A
                   ON S.codigo_articulo = A.c_articulo
            WHERE S.codigo_proveedor = :id_proveedor
              AND (:sucursales IS NULL OR S.codigo_sucursal = ANY(:sucursales))
              AND (:rubros     IS NULL OR A.c_rubro        = ANY(:rubros))
            ORDER BY S.codigo_articulo, S.codigo_sucursal
        """).bindparams(
            bindparam("id_proveedor", type_=INTEGER),
            bindparam("sucursales",   type_=ARRAY(INTEGER)),
            bindparam("rubros",       type_=ARRAY(INTEGER)),
        )

        print("[DEBUG] Ejecutando query de stock_sucursal...")
        stock_sucursal = pd.read_sql(
            sql_stock, conn,
            params={"id_proveedor": id_proveedor, "sucursales": sucursales, "rubros": rubros}
        )

        if stock_sucursal.empty:
            print(f"❗ No se encontraron artículos de Stock_Sucursal para el proveedor {id_proveedor}.")
            return None, None

        stock_sucursal = stock_sucursal.replace([np.inf, -np.inf], np.nan).fillna(0)
        stock_sucursal.columns = stock_sucursal.columns.str.upper()

        stock_sucursal['CODIGO_SUCURSAL'] = pd.to_numeric(stock_sucursal['CODIGO_SUCURSAL'], errors='coerce').astype('Int64')
        stock_sucursal['CODIGO_ARTICULO'] = pd.to_numeric(stock_sucursal['CODIGO_ARTICULO'], errors='coerce').astype('Int64')
        stock_sucursal['CODIGO_PROVEEDOR'] = pd.to_numeric(stock_sucursal['CODIGO_PROVEEDOR'], errors='coerce').astype('Int64')

        for c in ["PEDIDO_SGM","STOCK","PEDIDO_PENDIENTE","I_LISTA_CALCULADO","ULTIMO_INGRESO",
                  "PRECIO_VENTA","PRECIO_COSTO","TRANSFER_PENDIENTE","VENTA_UNIDADES_1Q","VENTA_UNIDADES_2Q"]:
            stock_sucursal[c] = pd.to_numeric(stock_sucursal[c], errors='coerce').astype('Float64')

        stock_sucursal['FACTOR_VENTA'] = pd.to_numeric(stock_sucursal['FACTOR_VENTA'], errors='coerce').astype('Int64')
        stock_sucursal['Q_DIAS_STOCK'] = pd.to_numeric(stock_sucursal['Q_DIAS_STOCK'], errors='coerce').astype('Int64')

        stock_sucursal.to_csv(archivo_stock, index=False, encoding='utf-8')
        print("---> Datos de Stock Sucursal guardados")

        # ------------------------
        # 4.3) VENTAS (con CTEs)
        # ------------------------
        sql_ventas = text("""
            WITH ventas_unificadas AS (
                SELECT v.f_venta AS fecha,
                       v.c_articulo AS codigo_articulo,
                       v.c_sucu_empr AS sucursal,
                       v.q_unidades_vendidas AS unidades
                FROM src.t702_est_vtas_por_articulo v
                WHERE v.f_venta >= :desde
                UNION ALL
                SELECT v.f_venta AS fecha,
                       v.c_articulo AS codigo_articulo,
                       v.c_sucu_empr AS sucursal,
                       v.q_unidades_vendidas AS unidades
                FROM src.t702_est_vtas_por_articulo_dbarrio v
                WHERE v.f_venta >= :desde
            ),
            articulos_filtrados AS (
                SELECT m.c_articulo
                FROM src.m_3_articulos m
                WHERE :rubros IS NULL OR m.c_rubro = ANY(:rubros)
            ),
            surtido_filtrado AS (
                SELECT a.c_articulo  AS codigo_articulo,
                       a.c_sucu_empr AS sucursal
                FROM src.base_productos_vigentes a
                WHERE a.c_proveedor_primario = :id_proveedor
                  AND (:sucursales IS NULL OR a.c_sucu_empr = ANY(:sucursales))
                  AND (:rubros IS NULL OR a.c_articulo IN (SELECT c_articulo FROM articulos_filtrados))
            )
            SELECT v.fecha, v.codigo_articulo, v.sucursal, v.unidades
            FROM ventas_unificadas v
            JOIN surtido_filtrado s
              ON s.codigo_articulo = v.codigo_articulo
             AND s.sucursal        = v.sucursal
            ORDER BY v.fecha, v.codigo_articulo, v.sucursal
        """).bindparams(
            bindparam("id_proveedor", type_=INTEGER),
            bindparam("desde"),
            bindparam("sucursales",   type_=ARRAY(INTEGER)),
            bindparam("rubros",       type_=ARRAY(INTEGER)),
        )

        params_ventas = build_params(
            id_proveedor=id_proveedor,
            desde="2024-06-01",
            sucursales=sucursales,
            rubros=rubros
        )

        print("[DEBUG] Ejecutando query de ventas...")
        ventas = pd.read_sql(sql_ventas, conn, params=params_ventas)

    # ------- fuera del with: merge y salidas -------
    if ventas.empty:
        print(f"⚠️ No se encontraron ventas DIARCO + BARRIO para el proveedor {id_proveedor}.")
        ventas = pd.DataFrame(columns=['fecha','codigo_articulo','sucursal','unidades'])

    if not ventas.empty:
        ventas.columns = ventas.columns.str.lower()
        ventas['sucursal'] = pd.to_numeric(ventas['sucursal'], errors='coerce').astype('Int64')
        ventas['codigo_articulo'] = pd.to_numeric(ventas['codigo_articulo'], errors='coerce').astype('Int64')
        ventas['fecha'] = pd.to_datetime(ventas['fecha'])
        ventas = ventas.dropna(subset=['fecha','codigo_articulo','sucursal'], how='all')

    ventas = ventas.rename(columns={
        "fecha": "Fecha",
        "codigo_articulo": "Codigo_Articulo",
        "sucursal": "Sucursal",
        "unidades": "Unidades"
    })
    ventas.to_csv(f'{folder}/{etiqueta}_Demanda.csv', index=False, encoding='utf-8')
    print("---> Datos de Ventas guardados")

    # --- MERGE FINAL ---
    data = pd.merge(
        articulos,
        ventas,
        left_on=['C_ARTICULO', 'C_SUCU_EMPR'],
        right_on=['Codigo_Articulo', 'Sucursal'],
        how='left'
    )

    if data.empty:
        print(f"⚠️ No hay coincidencias entre artículos y ventas para el proveedor {id_proveedor}.")
        return None, articulos

    # Casting seguro para columnas clave
    for c in ['C_ARTICULO', 'C_SUCU_EMPR', 'Codigo_Articulo', 'Sucursal']:
        data[c] = pd.to_numeric(data[c], errors='coerce').astype('Int64')

    data['TIENE_DEMANDA'] = data['Unidades'].notna().astype(int)
    data['Unidades'] = pd.to_numeric(data['Unidades'], errors='coerce').fillna(0)

    data.to_csv(archivo_datos, index=False, encoding='utf-8')
    print("---> Datos de RECUPERACIÓN guardados")

    # Compacto de ventas
    file_path = f'{folder}/{etiqueta}_Ventas.csv'
    print(f"[DEBUG] Ruta destino definida en .env: {folder}")
    ventas_compacto = data[['Fecha','Codigo_Articulo','Sucursal','Unidades']]
    ventas_compacto.to_csv(file_path, index=False, encoding='utf-8')
    print(f"---> Datos de Ventas guardados: {file_path}")

    return data, articulos


In [ ]:
name = "20_ARCOR"
algoritmo = "ALGO_05"
id_proveedor = 20
sucursales = [1, 2, 49, 302]   # listas (arrays)
rubros = [2055, 2053]          # listas (arrays)

data, articulos = get_forecast(
    id_proveedor, name, period_lengh=90,
    algorithm=algoritmo, f1=0.5, f2=0.3, f3=0.2,
    current_date=None,
    sucursales=sucursales, rubros=rubros
)


# PRUEBA de CONEXIONES

In [2]:
# diag_conexion_tabla.py
# Verifica conexión y existencia/permiso de una tabla con los mismos parámetros de su .env
# Requisitos: pip install python-dotenv sqlalchemy psycopg2-binary pandas

import os
from typing import Dict, Any, Tuple, Optional

import pandas as pd
from dotenv import dotenv_values

import psycopg2
from psycopg2.extras import RealDictCursor

import sqlalchemy
from sqlalchemy import text
from sqlalchemy.engine import URL


def load_secrets_from_env() -> Dict[str, Any]:
    """Carga el .env igual que su código productivo."""
    env_path = os.environ.get("FORECAST_ENV_PATH", "C:/ETL/FORECAST/.env")
    secrets = dotenv_values(env_path)
    required = ["PGP_HOST", "PGP_PORT", "PGP_DB", "PGP_USER", "PGP_PASSWORD"]
    missing = [k for k in required if not secrets.get(k)]
    if missing:
        raise RuntimeError(f"Faltan claves en {env_path}: {missing}")
    return secrets


def build_sqlalchemy_engine(secrets: Dict[str, Any]) -> sqlalchemy.Engine:
    """
    Construye el Engine asegurando el ESCAPADO de credenciales.
    Usamos el mismo host/port/db/user/password que psycopg2.
    """
    url = URL.create(
        drivername="postgresql+psycopg2",   # driver explícito y estable
        username=secrets["PG_USER"],
        password=secrets["PG_PASSWORD"],   # URL.create hace el quoting correcto
        host=secrets["PG_HOST"],
        port=int(secrets["PG_PORT"]),
        database=secrets["PG_DB"],
    )
    return sqlalchemy.create_engine(url, pool_pre_ping=True, future=True)


def build_psycopg2_conn(secrets: Dict[str, Any]) -> psycopg2.extensions.connection:
    """Crea la conexión psycopg2 con los mismos parámetros del .env."""
    return psycopg2.connect(
        host=secrets["PG_HOST"],
        port=int(secrets["PG_PORT"]),
        dbname=secrets["PG_DB"],
        user=secrets["PG_USER"],
        password=secrets["PG_PASSWORD"],
        cursor_factory=RealDictCursor,
    )


def _split_fqname(fqname: str) -> Tuple[str, str]:
    """
    'src.base_productos_vigentes' -> ('src', 'base_productos_vigentes')
    """
    parts = fqname.split(".")
    if len(parts) != 2:
        raise ValueError(f"Nombre de tabla no calificado: {fqname} (esperado esquema.tabla)")
    return parts[0], parts[1]


def fmt_row(row: Optional[dict]) -> str:
    return "NULL" if row is None else ", ".join(f"{k}={v}" for k, v in row.items())


WHOAMI_SQL = """
SELECT
  current_database()           AS db,
  current_user                 AS user,
  session_user                 AS session_user,
  inet_server_addr()::text     AS server_ip,
  inet_server_port()           AS server_port,
  version()                    AS version;
"""

SEARCH_PATH_SQL = "SHOW search_path;"

EXISTENCE_SQL = """
SELECT
  to_regclass(:fqname)                                           AS regclass,
  EXISTS (
    SELECT 1
    FROM information_schema.tables
    WHERE table_schema = :schema AND table_name = :table
  )                                                              AS in_information_schema,
  (SELECT has_schema_privilege(current_user, :schema, 'USAGE'))  AS has_schema_usage,
  (SELECT has_table_privilege(current_user, :fqname, 'SELECT'))  AS can_select
;
"""

def probe_sql(schema: str, table: str) -> str:
    return f"SELECT 1 FROM {schema}.{table} LIMIT 1;"


def verificar_sqlalchemy(secrets: Dict[str, Any], fq_table: str) -> None:
    schema, table = _split_fqname(fq_table)
    print("\n=== Verificación con SQLAlchemy ===")
    engine = build_sqlalchemy_engine(secrets)
    with engine.connect() as conn:
        who = conn.execute(text(WHOAMI_SQL)).mappings().first()
        print(f"[WHOAMI] {fmt_row(dict(who) if who else None)}")

        sp = conn.execute(text(SEARCH_PATH_SQL)).scalar()
        print(f"[search_path] {sp}")

        exist = conn.execute(
            text(EXISTENCE_SQL),
            {"fqname": fq_table, "schema": schema, "table": table},
        ).mappings().first()
        print(f"[existencia] {fmt_row(dict(exist) if exist else None)}")

        # Prueba de acceso real
        try:
            conn.execute(text(probe_sql(schema, table)))
            print("[probe] SELECT 1 ... OK")
        except Exception as e:
            print(f"[probe] ERROR -> {type(e).__name__}: {e}")


def verificar_psycopg2(secrets: Dict[str, Any], fq_table: str) -> None:
    schema, table = _split_fqname(fq_table)
    print("\n=== Verificación con psycopg2 ===")
    with build_psycopg2_conn(secrets) as conn:
        with conn.cursor() as cur:
            cur.execute(WHOAMI_SQL)
            who = cur.fetchone()
            print(f"[WHOAMI] {fmt_row(who)}")

            cur.execute(SEARCH_PATH_SQL)
            sp = cur.fetchone()
            print(f"[search_path] {sp.get('search_path') if sp else 'NULL'}")

            cur.execute(
                """
                SELECT
                  to_regclass(%s) AS regclass,
                  EXISTS (
                    SELECT 1 FROM information_schema.tables
                    WHERE table_schema = %s AND table_name = %s
                  ) AS in_information_schema,
                  has_schema_privilege(current_user, %s, 'USAGE') AS has_schema_usage,
                  has_table_privilege(current_user, %s, 'SELECT') AS can_select
                """,
                (fq_table, schema, table, schema, fq_table)
            )
            exist = cur.fetchone()
            print(f"[existencia] {fmt_row(exist)}")

            try:
                cur.execute(probe_sql(schema, table))
                _ = cur.fetchone()
                print("[probe] SELECT 1 ... OK")
            except Exception as e:
                print(f"[probe] ERROR -> {type(e).__name__}: {e}")


def diagnostico(fq_table: str = "src.base_productos_vigentes") -> None:
    """Ejecuta la verificación con ambos conectores usando el .env de producción."""
    secrets = load_secrets_from_env()

    # Mostrar parámetros efectivos (sin password)
    print("=== Parámetros de conexión cargados del .env ===")
    print(f"Host={secrets['PGP_HOST']}  Port={secrets['PGP_PORT']}  DB={secrets['PGP_DB']}  User={secrets['PGP_USER']}")
    print(f"Tabla objetivo: {fq_table}")

    try:
        verificar_sqlalchemy(secrets, fq_table)
    except Exception as e:
        print(f"\n[SQLAlchemy] ERROR -> {type(e).__name__}: {e}")

    try:
        verificar_psycopg2(secrets, fq_table)
    except Exception as e:
        print(f"\n[psycopg2] ERROR -> {type(e).__name__}: {e}")

    print("\n=== Pistas de diagnóstico si difieren ===")
    print("- Si [WHOAMI] o server_ip/server_port difieren entre conectores, están apuntando a servidores distintos.")
    print("- Si 'regclass' es NULL pero 'in_information_schema' es True, podría haber tema de comillas/caso del nombre.")
    print("- Si 'has_schema_usage' = False, falta USAGE sobre el esquema (p.ej. 'src').")
    print("- Si 'can_select' = False, falta SELECT sobre la tabla.")
    print("- Si el probe falla solo en SQLAlchemy, revisen contraseñas con caracteres especiales: usen URL.create (como aquí).")
    print("- Si todo está OK en psycopg2 y falla en SQLAlchemy, comparen SEARCH_PATH; a veces hay GUCs aplicadas por cliente.")



In [3]:
# if __name__ == "__main__":
#     # Ejemplo de uso:
#     try:
#         from secrets_local import secrets  # o importen su dict real
#     except Exception:
#         print("⚠️ Deben proveer el dict 'secrets' con PGP_HOST/PORT/DB/USER/PASSWORD.")
#         sys.exit(1)

diagnostico( fq_table="src.base_productos_vigentes")

=== Parámetros de conexión cargados del .env ===
Host=186.158.182.54  Port=5432  DB=connexa_platform  User=postgres
Tabla objetivo: src.base_productos_vigentes

=== Verificación con SQLAlchemy ===
[WHOAMI] db=diarco_data, user=postgres, session_user=postgres, server_ip=10.1.100.10/32, server_port=5432, version=PostgreSQL 14.18 (Ubuntu 14.18-0ubuntu0.22.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0, 64-bit
[search_path] "$user", public
[existencia] regclass=src.base_productos_vigentes, in_information_schema=True, has_schema_usage=True, can_select=True
[probe] SELECT 1 ... OK

=== Verificación con psycopg2 ===
[WHOAMI] db=diarco_data, user=postgres, session_user=postgres, server_ip=10.1.100.10/32, server_port=5432, version=PostgreSQL 14.18 (Ubuntu 14.18-0ubuntu0.22.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0, 64-bit
[search_path] "$user", public
[existencia] regclass=src.base_productos_vigentes, in_information_

## INTEGRACIÓN CONTINUA (CI/CD) con GITLAB

In [ ]:
# Selección del algoritmo de predicción
match algorithm:
    case 'ALGO_01':
        return Procesar_ALGO_01(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date, factor_last=f1, factor_previous=f2, factor_year=f3)  # Promedio Ponderado x 3 Factores
    case 'ALGO_02':
        return Procesar_ALGO_02(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date) # Doble Exponencial - Modelo Holt (Tendencia)
    case 'ALGO_03':
        return Procesar_ALGO_03(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date, periodos=f1, estacionalidad=f2, tendencia=f3) # Triple Exponencial Holt-WInter (Tendencia + Estacionalidad) (periodos, add, add)
    case 'ALGO_04':
        return Procesar_ALGO_04(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date, alpha=f1) # EWMA con Factor alpha
    case 'ALGO_05':
        return Procesar_ALGO_05(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date) # Promedio Venta Simple en Ventana
    case 'ALGO_06':
        return Procesar_ALGO_06(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date) # Tendencias Ventas Semanales
    case 'ALGO_07':
        return Procesar_ALGO_07(data, articulos, id_proveedor, lbl_proveedor, period_lengh, current_date, factor=f1, fecha_base=f2)  # Promedio Simple Ventana Base Movil x Factor
    case _:
        raise ValueError(f"Error: El algoritmo '{algorithm}' no está implementado.")
